In [1]:
import sys
sys.path.append("..")
from src.ingest.build import load_raw
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

df_x, df_y, df_z = load_raw()

series = {}
for nombre, df in [("X", df_x), ("Y", df_y), ("Z", df_z)]:
    series[nombre] = df.set_index("Date")[f"Price_{nombre}"].sort_index().resample("ME").mean()

inicio = max(s.index.min() for s in series.values())
series = {k: s[s.index >= inicio] for k, s in series.items()}

for nombre, s in series.items():
    print(nombre, s.index.min().date(), "→", s.index.max().date(), len(s))
fin_comun = min(s.index.max() for s in series.values())
corte = fin_comun - pd.DateOffset(months=12)
print("fin común:", fin_comun.date(), "| corte:", corte.date())

X 2010-01-31 → 2024-04-30 172
Y 2010-01-31 → 2023-09-30 165
Z 2010-01-31 → 2023-08-31 164
fin común: 2023-08-31 | corte: 2022-08-31


In [3]:
from src.forecast.commodities import backtest, ranking

cortes = pd.to_datetime(["2019-08-31", "2020-08-31", "2021-08-31", "2022-08-31"])
bt = backtest(series, cortes)
rk = ranking(bt)
rk.round(4)

21:51:10 - cmdstanpy - INFO - Chain [1] start processing
21:51:10 - cmdstanpy - INFO - Chain [1] done processing
21:51:10 - cmdstanpy - INFO - Chain [1] start processing
21:51:10 - cmdstanpy - INFO - Chain [1] done processing
21:51:10 - cmdstanpy - INFO - Chain [1] start processing
21:51:10 - cmdstanpy - INFO - Chain [1] done processing
21:51:11 - cmdstanpy - INFO - Chain [1] start processing
21:51:11 - cmdstanpy - INFO - Chain [1] done processing
21:51:11 - cmdstanpy - INFO - Chain [1] start processing
21:51:11 - cmdstanpy - INFO - Chain [1] done processing
21:51:11 - cmdstanpy - INFO - Chain [1] start processing
21:51:11 - cmdstanpy - INFO - Chain [1] done processing
21:51:11 - cmdstanpy - INFO - Chain [1] start processing
21:51:12 - cmdstanpy - INFO - Chain [1] done processing
21:51:12 - cmdstanpy - INFO - Chain [1] start processing
21:51:12 - cmdstanpy - INFO - Chain [1] done processing
21:51:12 - cmdstanpy - INFO - Chain [1] start processing
21:51:12 - cmdstanpy - INFO - Chain [1]

,serie,h,modelo,MAPE,RMSE,MASE,ganador
3,X,3,lightgbm,0.0595,4.9412,0.9642,True
4,X,3,naive,0.0686,5.3425,1.1666,False
2,X,3,ets,0.0679,5.4063,1.1760,False
1,X,3,drift,0.0679,5.4063,1.1760,False
0,X,3,arima,0.0836,6.0930,1.3713,False
...,...,...,...,...,...,...,...
62,Z,12,sarima,0.1058,316.0181,3.7926,False
56,Z,12,arima,0.1077,314.7844,3.8192,False
58,Z,12,ets,0.1077,314.1353,3.8519,False
57,Z,12,drift,0.1077,314.1362,3.8519,False


In [7]:
rk[(rk.serie == "X") & (rk.h == 12)].round(4)

,serie,h,modelo,MAPE,RMSE,MASE,ganador
17,X,12,lightgbm,0.2470,20.0439,3.8674,True
14,X,12,arima,0.2473,19.3803,3.9501,False
18,X,12,naive,0.2529,19.7835,3.9685,False
20,X,12,sarima,0.2483,19.6468,4.0077,False
16,X,12,ets,0.2568,20.3279,4.0798,False
15,X,12,drift,0.2568,20.3279,4.0798,False
19,X,12,prophet,0.3193,26.5075,5.7703,False


In [8]:
rk[rk.h == 3].round(4)

,serie,h,modelo,MAPE,RMSE,MASE,ganador
3,X,3,lightgbm,0.0595,4.9412,0.9642,True
4,X,3,naive,0.0686,5.3425,1.1666,False
2,X,3,ets,0.0679,5.4063,1.1760,False
1,X,3,drift,0.0679,5.4063,1.1760,False
0,X,3,arima,0.0836,6.0930,1.3713,False
6,X,3,sarima,0.0895,6.6442,1.4771,False
5,X,3,prophet,0.2381,18.5130,4.2675,False
21,Y,3,arima,0.0572,41.3030,1.4948,True
27,Y,3,sarima,0.0620,43.4683,1.5918,False
25,Y,3,naive,0.0635,45.4677,1.6506,False


In [6]:
rk[rk['ganador'] == True]

,serie,h,modelo,MAPE,RMSE,MASE,ganador
3,X,3,lightgbm,0.059528,4.941200,0.964197,True
10,X,6,lightgbm,0.091279,8.176590,1.505310,True
17,X,12,lightgbm,0.247016,20.043859,3.867391,True
21,Y,3,arima,0.057189,41.302970,1.494759,True
34,Y,6,sarima,0.080562,64.027440,2.364869,True
41,Y,12,sarima,0.134950,119.621759,4.484053,True
48,Z,3,sarima,0.044786,123.271241,1.562996,True
53,Z,6,naive,0.062629,182.958123,2.258889,True
59,Z,12,lightgbm,0.098387,304.718172,3.671436,True


#### Selección de modelo y horizonte

Se evaluaron siete modelos sobre cuatro ventanas de corte (2019 a 2022) y tres horizontes
(3, 6 y 12 meses), con MAPE, RMSE y MASE. Se descarta R², no es relevante para pronóstico de series de tiempo.

##### Horizonte

El error con el horizonte en las tres series.

| Serie | MAPE h=3 | MAPE h=12 |
|---|---|---|
| X | 6.0% | 24.7% |
| Y | 5.7% | 13.5% |
| Z | 4.5% | 9.8% |

A 12 meses ningún modelo se distingue del resto. Todos quedan entre 24.7% y 25.7% en X, y el
MASE supera 3.8, es decir, el error es casi cuatro veces el cambio típico mes a mes. Se adopta
un horizonte de 3 meses como base de planeación, con 6 meses como escenario extendido de mayor
incertidumbre.

##### Modelo por serie (h=3)

| Serie | Modelo | MAPE | MASE | Criterio |
|---|---|---|---|---|
| X | LightGBM | 5.95% | 0.96 | Único modelo con MASE menor a 1 en todo el experimento. Mejora 13% sobre naive |
| Y | Naive | 6.35% | 1.65 | Ningún modelo supera al benchmark de forma consistente. ARIMA gana en MASE y LightGBM en MAPE, ambos por margen mínimo. Se elige el más simple por parsimonia |
| Z | SARIMA(1,1,1)(1,0,1,12) | 4.48% | 1.56 | Mejora 10% sobre naive y es consistente en las tres métricas |

La selección no toma el mínimo de forma mecánica. Cuando las diferencias entre modelos están
dentro del ruido, se prefiere el modelo más simple. Prophet queda descartado en las tres series,
con MAPE entre 17% y 24%, probablemente por imponer estacionalidad anual a series que no la tienen.